# 04 · From activity cliffs to plausible metabolic transformations

**OpenADMET CYP Challenge — chemical interpretation and modeling study**

**Question:** Are some apparent CYP inhibition activity cliffs better understood as differences between chemically related compounds—for example, a putative aromatic hydroxylation product or the carboxylate derived from a methyl ester—than as inexplicable changes between similar structures?

This Colab notebook is **standalone**; earlier notebooks are not required. It downloads the public **training** data, looks up specific OCNT identifiers and SMILES, tests *structural transformation hypotheses*, searches for other exact structure-related pairs, compares the CYP activities and experimental confidence intervals, and benchmarks alternative molecular representations and simple regressors.

### Chemist-proposed case studies

| Proposed precursor | Proposed product | Structural hypothesis | Interpretation to test |
|---|---|---|---|
| `OCNT-2328565` | `OCNT-2328962` | Addition of phenolic OH to an aromatic ring | Plausible CYP oxidative hydroxylation **if the structures match** |
| `OCNT-2395476` | `OCNT-2395467` | Conversion of methyl ester to carboxylic acid/carboxylate | Plausible esterase hydrolysis or intentional ester/acid analog synthesis |

**Important distinctions:** A plausible one-step transformation is **not evidence that metabolism occurred in the experiment**. A methyl ester is **not established as a prodrug**, nor the acid as an active drug, without pharmacological and metabolic data. CYP inhibition measured in the released data does not by itself demonstrate formation of either metabolite. Treat the proposed directions as hypotheses until literature or experiments verify them.

The generated files are written to `cyp_notebook_04_results/` and bundled at the end for convenient download.

## 1 · Install required packages

CPU execution is sufficient. Choose **Runtime → Run all** in Google Colab. The PubChem API section is optional; the rest of the notebook does not depend on PubChem availability.

In [ ]:
%pip -q install "rdkit>=2023.9" "lightgbm>=4.0" "scikit-learn>=1.3" "scipy>=1.11" "pandas>=2.0" "numpy>=1.24" "matplotlib>=3.7" "requests>=2.28"

## 2 · Configuration and imports

In [ ]:
import os, json, platform, warnings, time
from pathlib import Path
from urllib.parse import quote
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display, Markdown, Image
from rdkit import Chem, DataStructs, RDLogger, rdBase
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFMCS, Draw
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr
from lightgbm import LGBMRegressor

RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.info')
SEED = 2026
OUTPUT = Path(os.environ.get('CYP_NB04_OUTPUT', 'cyp_notebook_04_results'))
OUTPUT.mkdir(parents=True,exist_ok=True)
DATA_URL = 'https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/cyp-challenge-TRAIN_inhibition.csv'
LOCAL_CSV = os.environ.get('CYP_DATA_PATH')  # optional local CSV override for offline testing
ISO = ('CYP1A2','CYP2C9','CYP2D6','CYP3A4')
SIM_THRESHOLDS = (0.55,0.65,0.75)
CLIFF_DELTA = 1.0
RUN_PUBCHEM_LOOKUP = False   # set True to query PubChem live after inspecting the SMILES
RUN_MODEL_COMPARISON = True  # turn False for a chemistry-only quick run
RF_TREES = 120
GBM_TREES = 200
MAX_DRAW_PAIRS = 8

if os.environ.get('CYP_NB04_TEST') == '1':
    RF_TREES=12
    GBM_TREES=20
print('Python:',platform.python_version(),'| RDKit:',rdBase.rdkitVersion,
      '| sklearn:',__import__('sklearn').__version__)

## 3 · Load the data, verify compound identifiers, extract exact SMILES

**Requested SMILES snippet:** The cell prints the original, unmodified dataset SMILES for `OCNT-2395476` (and the other three compounds). The raw SMILES is what you should copy when searching PubChem; the normalized comparison SMILES is generated separately and may intentionally neutralize carboxylate charge or remove counterions.

In [ ]:
raw = pd.read_csv(LOCAL_CSV or DATA_URL)
for field in ('Molecule_Name','SMILES'):
    if field not in raw.columns:
        raise ValueError(f'Missing expected column {field}: {raw.columns.tolist()}')
raw = raw.copy()
raw['Molecule_Name'] = raw['Molecule_Name'].astype(str).str.strip().str.upper()
print(f'Loaded {len(raw):,} rows and {raw.Molecule_Name.nunique():,} unique named compounds')
print('Columns:',raw.columns.tolist())

COMPOUNDS = ['OCNT-2395476', 'OCNT-2395467', 'OCNT-2328565', 'OCNT-2328962']
lookup = raw.loc[raw.Molecule_Name.isin(COMPOUNDS), ['Molecule_Name','SMILES']].drop_duplicates()
lookup = lookup.set_index('Molecule_Name').reindex(COMPOUNDS).reset_index()
print('Requested dataset SMILES values (do not silently invent missing entries):')
display(lookup)
for name in COMPOUNDS:
    matched=lookup.loc[lookup.Molecule_Name==name, 'SMILES'].dropna()
    if matched.empty:
        print(f'{name}: NOT FOUND in direct-inhibition training CSV')
    else:
        smi=str(matched.iloc[0]); print(f'{name} SMILES: {smi}')
        url='https://pubchem.ncbi.nlm.nih.gov/#query='+quote(smi,safe='')
        display(Markdown(f'[{name}: search structure in PubChem]({url})'))
lookup.to_csv(OUTPUT/'named_compound_smiles.csv', index=False)

### Optional: ask PubChem for matching CIDs by **structure**

PubChem does not necessarily recognize an internal `OCNT-...` name. Submit its **SMILES** using the official PUG REST API. A returned CID is a **structure association**, not proof of shared pharmacological activity or that a metabolite has been reported. If no CID appears, do not infer that the compound is novel. Always verify stereochemistry, protonation, salt forms, and synonyms on the linked PubChem record.

Set `RUN_PUBCHEM_LOOKUP = True` in Section 2 and rerun the following cell to execute network requests. Network outages and absent PubChem matches are handled gracefully.

In [ ]:
def pubchem_cids_from_smiles(smiles, timeout=18):
    endpoint='https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/cids/JSON'
    try:
        resp=requests.post(endpoint,data={'smiles':str(smiles)},timeout=timeout,
                           headers={'User-Agent':'CYP-uncertainty-education/0.1'})
        if resp.status_code==404:
            return [],'No indexed exact-structure match reported'
        resp.raise_for_status()
        return resp.json().get('IdentifierList',{}).get('CID',[]),'OK'
    except (requests.RequestException,ValueError) as exc:
        return [],f'Lookup unavailable: {type(exc).__name__}: {str(exc)[:120]}'

pubchem_rows=[]
if RUN_PUBCHEM_LOOKUP:
    for row in lookup.dropna(subset=['SMILES']).itertuples(index=False):
        cids,status=pubchem_cids_from_smiles(row.SMILES)
        pubchem_rows.append({'Molecule_Name':row.Molecule_Name,'CIDs':','.join(map(str,cids)),'status':status})
        if cids: display(Markdown(f'[{row.Molecule_Name} → PubChem CID {cids[0]}](https://pubchem.ncbi.nlm.nih.gov/compound/{cids[0]})'))
        print(row.Molecule_Name, cids[:5],status)
else:
    print('PubChem API lookup disabled by default; structure-search links are printed above.')
if pubchem_rows: pd.DataFrame(pubchem_rows).to_csv(OUTPUT/'pubchem_structure_hits.csv',index=False)

## 4 · Standardize structures **only for structural relationship testing**

We keep the **original** SMILES for reporting and retain original RDKit molecules for descriptors and prediction. For testing whether a proposed neutral acid matches a listed carboxylate, we additionally remove counterion fragments and neutralize suitable ionized functionalities. This can collapse ionization-state distinctions, so both raw and normalized comparisons are retained and should not be conflated.

In [ ]:
chooser=rdMolStandardize.LargestFragmentChooser()
uncharger=rdMolStandardize.Uncharger()

def normalized_molecule(mol):
    if mol is None: return None
    try:
        m=chooser.choose(Chem.Mol(mol))
        m=uncharger.uncharge(m)
        Chem.SanitizeMol(m)
        return m
    except Exception:
        return None

def normalized_smiles(mol):
    m=normalized_molecule(mol)
    return Chem.MolToSmiles(m,isomericSmiles=True) if m is not None else None

raw['_mol']=[Chem.MolFromSmiles(s) if isinstance(s,str) else None for s in raw.SMILES]
work=raw.loc[raw['_mol'].notna()].copy().reset_index(drop=True)
work['_normalized_smiles']=[normalized_smiles(m) for m in work['_mol']]
work['_original_canonical']=[Chem.MolToSmiles(m,isomericSmiles=True) for m in work['_mol']]
name_to_idx={n:i for i,n in enumerate(work.Molecule_Name)}
key_to_indices={}
for i,key in enumerate(work['_normalized_smiles']):
    if key is not None: key_to_indices.setdefault(key,[]).append(i)
print('Valid structures:',len(work),'| unique normalized structures:',len(key_to_indices))
for name in COMPOUNDS:
    if name in name_to_idx:
        i=name_to_idx[name]
        print(f'{name:14s} raw={work.loc[i,"SMILES"]}')
        print(f'{" ":14s} normalized={work.loc[i,"_normalized_smiles"]}')

## 5 · Generate the proposed transformations by chemistry rules

The code tests **exact structural changes**, not predicted enzyme selectivity or reaction rates:

- **Aromatic hydroxylation:** add OH to an aromatic carbon that originally has a hydrogen (`Ar–H → Ar–OH`). This is *consistent with* a CYP oxidation, but the reaction need not occur in an assay.
- **Methyl-ester hydrolysis:** `R–C(=O)–OCH₃ → R–C(=O)–OH` (and normalize an acid/carboxylate pair for comparison). A carboxylic acid could be independently synthesized; this rule does not demonstrate an esterase or prodrug relationship.
- **Aryl O-demethylation** (exploratory): `Ar–O–CH₃ → Ar–OH`.

Only direct, single-step graph transformations matching one of these explicit rules are annotated. Other biochemical transformations are outside scope. The rules do not predict whether a transformation occurs biologically.

In [ ]:
ESTER_METHYL=Chem.MolFromSmarts('[CX3](=[OX1])[OX2][CH3]')
ARYL_ETHER_METHYL=Chem.MolFromSmarts('[c][OX2][CH3]')

def remove_terminal_methyl(mol, methyl_idx):
    rw=Chem.RWMol(mol)
    rw.RemoveAtom(int(methyl_idx))
    out=rw.GetMol()
    Chem.SanitizeMol(out)
    return out

def enumerate_candidate_transformations(mol):
    """Return proposed single-step product structures, as (label, normalized_smiles)."""
    if mol is None: return set()
    candidates=set()
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum()==6 and atom.GetIsAromatic() and atom.GetTotalNumHs()>0:
            try:
                rw=Chem.RWMol(mol)
                oi=rw.AddAtom(Chem.Atom(8))
                rw.AddBond(atom.GetIdx(),oi,Chem.BondType.SINGLE)
                new=rw.GetMol(); Chem.SanitizeMol(new)
                key=normalized_smiles(new)
                if key: candidates.add(('aromatic_hydroxylation',key))
            except (ValueError,RuntimeError):
                pass
    for match in mol.GetSubstructMatches(ESTER_METHYL):
        try:
            key=normalized_smiles(remove_terminal_methyl(mol,match[-1]))
            if key: candidates.add(('methyl_ester_hydrolysis',key))
        except (ValueError,RuntimeError):
            pass
    for match in mol.GetSubstructMatches(ARYL_ETHER_METHYL):
        try:
            key=normalized_smiles(remove_terminal_methyl(mol,match[-1]))
            if key: candidates.add(('aryl_O_demethylation',key))
        except (ValueError,RuntimeError):
            pass
    return candidates

def detect_proposed_relationship(parent_mol,product_mol):
    if parent_mol is None or product_mol is None: return []
    target=normalized_smiles(product_mol)
    return sorted({label for label,key in enumerate_candidate_transformations(parent_mol) if key==target})

# Known transformation unit checks, including an ionized carboxylate.
assert 'aromatic_hydroxylation' in detect_proposed_relationship(
    Chem.MolFromSmiles('c1ccccc1'),Chem.MolFromSmiles('Oc1ccccc1'))
assert 'methyl_ester_hydrolysis' in detect_proposed_relationship(
    Chem.MolFromSmiles('COC(=O)c1ccccc1'),Chem.MolFromSmiles('O=C([O-])c1ccccc1'))
assert 'aryl_O_demethylation' in detect_proposed_relationship(
    Chem.MolFromSmiles('COc1ccccc1'),Chem.MolFromSmiles('Oc1ccccc1'))
print('Three transformation checks passed.')

## 6 · Investigate the two chemist-identified compound pairs

The code respects the *proposed* precursor → product direction. A rule match supports only **structural compatibility** with that reaction. We also report molecular formula and descriptor differences, pIC50 measurements where both were assayed for a CYP, and experimental confidence-interval separation. The activity and CI findings cannot establish that a compound is a metabolite, prodrug, or active species.

In [ ]:
CASE_STUDIES=[
    {'parent':'OCNT-2328565','product':'OCNT-2328962','proposed':'aromatic_hydroxylation'},
    {'parent':'OCNT-2395476','product':'OCNT-2395467','proposed':'methyl_ester_hydrolysis'},
]

def physicochemical(mol):
    if mol is None: return {}
    return {'formula':rdMolDescriptors.CalcMolFormula(mol),
            'mw':Descriptors.MolWt(mol),
            'logp':Descriptors.MolLogP(mol),
            'tpsa':Descriptors.TPSA(mol),
            'hbd':rdMolDescriptors.CalcNumHBD(mol),
            'hba':rdMolDescriptors.CalcNumHBA(mol),
            'formal_charge':Chem.GetFormalCharge(mol),
            'inchikey':Chem.MolToInchiKey(mol)}

def assay_pair_rows(idx_a,idx_b,source='curated'):
    rows=[]
    for iso in ISO:
        col=f'{iso}_pIC50_direct_inhibition'
        if col not in work.columns: continue
        y1=pd.to_numeric(work.loc[idx_a,col],errors='coerce')
        y2=pd.to_numeric(work.loc[idx_b,col],errors='coerce')
        low_col=col+'_conf_low'; high_col=col+'_conf_high'
        vals=[]
        for idx in (idx_a,idx_b):
            lo=pd.to_numeric(work.loc[idx,low_col],errors='coerce') if low_col in work else np.nan
            hi=pd.to_numeric(work.loc[idx,high_col],errors='coerce') if high_col in work else np.nan
            vals.append((lo,hi))
        gap=(max(0.,float(vals[0][0]-vals[1][1]),float(vals[1][0]-vals[0][1]))
             if all(np.isfinite(x) for interval in vals for x in interval) else np.nan)
        rows.append({'source':source,'parent':work.loc[idx_a,'Molecule_Name'],
                     'product':work.loc[idx_b,'Molecule_Name'],'isoform':iso,
                     'parent_pic50':y1,'product_pic50':y2,
                     'delta_product_minus_parent':y2-y1 if pd.notna(y1) and pd.notna(y2) else np.nan,
                     'abs_delta_pic50':abs(y2-y1) if pd.notna(y1) and pd.notna(y2) else np.nan,
                     'ci_gap':gap,
                     'both_measured':bool(pd.notna(y1) and pd.notna(y2))})
    return rows

case_rows=[]; case_assays=[]
for item in CASE_STUDIES:
    a_name,b_name=item['parent'],item['product']
    if a_name not in name_to_idx or b_name not in name_to_idx:
        case_rows.append({**item,'structural_match':'ID missing from this dataset',
                          'matched_rules':'','notes':'Search other released files or verify identifier'})
        print('Missing compound(s) for',a_name,b_name)
        continue
    a,b=name_to_idx[a_name],name_to_idx[b_name]
    am,bm=work.loc[a,'_mol'],work.loc[b,'_mol']
    rules=detect_proposed_relationship(am,bm)
    prop_a,prop_b=physicochemical(am),physicochemical(bm)
    na,nb=physicochemical(normalized_molecule(am)),physicochemical(normalized_molecule(bm))
    row={**item,'matched_rules':';'.join(rules),
         'structural_match':str(item['proposed'] in rules),
         'parent_smiles':work.loc[a,'SMILES'],'product_smiles':work.loc[b,'SMILES'],
         'parent_formula':prop_a['formula'],'product_formula':prop_b['formula'],
         'parent_inchikey':prop_a['inchikey'],'product_inchikey':prop_b['inchikey'],
         'delta_mw_raw':prop_b['mw']-prop_a['mw'],
         'delta_mw_neutralized':nb['mw']-na['mw'],
         'delta_logp_raw':prop_b['logp']-prop_a['logp'],
         'delta_logp_neutralized':nb['logp']-na['logp'],
         'delta_tpsa':prop_b['tpsa']-prop_a['tpsa'],
         'delta_hbd':prop_b['hbd']-prop_a['hbd'],
         'delta_formal_charge':prop_b['formal_charge']-prop_a['formal_charge']}
    case_rows.append(row)
    case_assays.extend(assay_pair_rows(a,b))
    print(f"{a_name} → {b_name}: matched rules={rules or ['none']}; "
          f"proposed={item['proposed']}; neutralized ΔMW={row['delta_mw_neutralized']:.2f}; "
          f"neutralized ΔlogP={row['delta_logp_neutralized']:.2f}")
case_df=pd.DataFrame(case_rows)
case_assay_df=pd.DataFrame(case_assays)
display(case_df.drop(columns=['parent_smiles','product_smiles'],errors='ignore'))
if not case_assay_df.empty: display(case_assay_df)
case_df.to_csv(OUTPUT/'curated_transformation_cases.csv',index=False)
case_assay_df.to_csv(OUTPUT/'curated_pair_cyp_activity.csv',index=False)

## 7 · Visualize each case, highlighting structural changes

Purple highlighting indicates atoms **not included in one selected maximum common substructure (MCS) match**. It is a depiction aid, not atom mapping or proof of reaction mechanism. Symmetric structures may have multiple equivalent MCS matches.

In [ ]:
def draw_comparison(mol_a,mol_b,legend_a,legend_b,path):
    mcs=rdFMCS.FindMCS([mol_a,mol_b],timeout=4,
                       atomCompare=rdFMCS.AtomCompare.CompareElements,
                       bondCompare=rdFMCS.BondCompare.CompareOrder,
                       ringMatchesRingOnly=True,completeRingsOnly=True)
    query=Chem.MolFromSmarts(mcs.smartsString) if mcs.smartsString else None
    match_a=mol_a.GetSubstructMatch(query) if query is not None else ()
    match_b=mol_b.GetSubstructMatch(query) if query is not None else ()
    diff_a=[a.GetIdx() for a in mol_a.GetAtoms() if a.GetIdx() not in set(match_a)]
    diff_b=[a.GetIdx() for a in mol_b.GetAtoms() if a.GetIdx() not in set(match_b)]
    png=Draw.MolsToGridImage([mol_a,mol_b],molsPerRow=2,subImgSize=(510,350),
                             legends=[legend_a,legend_b],
                             highlightAtomLists=[diff_a,diff_b],
                             returnPNG=True)
    if isinstance(png,bytes):
        Path(path).write_bytes(png)
    elif hasattr(png,'data') and isinstance(png.data,bytes):
        Path(path).write_bytes(png.data)
    elif hasattr(png,'save'):
        png.save(str(path))
    else:
        raise TypeError(f'Unsupported image output: {type(png)}')
    return {'mcs_atoms':int(mcs.numAtoms),'unmatched_atoms_parent':len(diff_a),
            'unmatched_atoms_product':len(diff_b)}

for idx,item in enumerate(CASE_STUDIES,1):
    if item['parent'] not in name_to_idx or item['product'] not in name_to_idx: continue
    a,b=name_to_idx[item['parent']],name_to_idx[item['product']]
    path=OUTPUT/f'case_{idx:02d}_{item["parent"]}_{item["product"]}.png'
    info=draw_comparison(work.loc[a,'_mol'],work.loc[b,'_mol'],
              f'Proposed precursor {item["parent"]}',
              f'Proposed product {item["product"]}',path)
    print(item['proposed'],info)
    display(Image(filename=str(path)))

## 8 · Search the entire training set for *exactly* related transformation products

Generate each supported structural transformation for every training molecule, and look for its exact normalized product in the same dataset. **Unlike a high-Tanimoto-only scan, this does not discard a true single-step analog merely because its fingerprint similarity is below 0.75.** Direction is defined by the explicit rule and remains chemically hypothetical.

For every matched structural pair, report which CYPs have **both** direct-inhibition labels. Report cliff rates conditional on having a measured pair—do not use all possible molecular pairs as the denominator for this analysis.

In [ ]:
start=time.time()
edges=[]; seen=set()
for parent_idx,mol in enumerate(work['_mol']):
    for label,key in enumerate_candidate_transformations(mol):
        for product_idx in key_to_indices.get(key,[]):
            if parent_idx == product_idx: continue
            record_key=(parent_idx,product_idx,label)
            if record_key in seen: continue
            seen.add(record_key)
            edges.append({'parent_idx':parent_idx,'product_idx':product_idx,
                          'parent':work.loc[parent_idx,'Molecule_Name'],
                          'product':work.loc[product_idx,'Molecule_Name'],
                          'transformation':label})
edges_df=pd.DataFrame(edges,columns=['parent_idx','product_idx','parent','product','transformation'])
print(f'Identified {len(edges_df)} directed structural transformation matches in {time.time()-start:.1f}s')
display(edges_df.groupby('transformation').size().rename('n_matches').reset_index() if len(edges_df) else edges_df)
edges_df.to_csv(OUTPUT/'detected_structural_transformation_pairs.csv',index=False)

edge_activity=[]
for e in edges_df.itertuples(index=False):
    for a in assay_pair_rows(e.parent_idx,e.product_idx,source='screened_structural_match'):
        a['transformation']=e.transformation
        edge_activity.append(a)
edge_activity_df=pd.DataFrame(edge_activity)
if len(edge_activity_df):
    edge_activity_df['is_point_cliff']=(edge_activity_df['both_measured'] &
        (edge_activity_df['abs_delta_pic50']>=CLIFF_DELTA))
    edge_activity_df['ci_nonoverlap']=edge_activity_df['both_measured'] & (edge_activity_df['ci_gap']>0)
    both=edge_activity_df.loc[edge_activity_df.both_measured].copy()
    print('Transformation-related pairs measured in at least one isoform:',
          both[['parent','product','transformation']].drop_duplicates().shape[0])
    summary=(both.groupby(['transformation','isoform'],as_index=False)
             .agg(n_labeled_pairs=('abs_delta_pic50','size'),
                  n_large_deltas=('is_point_cliff','sum'),
                  mean_abs_delta=('abs_delta_pic50','mean'),
                  n_disjoint_ci=('ci_nonoverlap','sum')))
else:
    both=edge_activity_df.copy()
    summary=pd.DataFrame(columns=['transformation','isoform','n_labeled_pairs','n_large_deltas','mean_abs_delta','n_disjoint_ci'])
display(summary)
edge_activity_df.to_csv(OUTPUT/'transformation_pair_activity_all_isoforms.csv',index=False)
summary.to_csv(OUTPUT/'transformation_activity_summary.csv',index=False)

## 9 · Descriptive comparison of transformation-related activity differences

These plots describe only same-isoform **labeled structural pairs**. Their frequencies are influenced by which molecules were selected for dose-response experiments and by the support of our three narrowly defined chemistry rules. This is not a population-level estimate of metabolism frequency.

In [ ]:
if not summary.empty:
    panel=summary.pivot(index='isoform',columns='transformation',values='n_labeled_pairs').fillna(0)
    ax=panel.plot(kind='bar',figsize=(10,4.5))
    ax.set_ylabel('Number of measured transformation-related pairs')
    ax.set_title('CYP direct-inhibition labels available for structural transformation pairs')
    plt.xticks(rotation=0);plt.tight_layout()
    plt.savefig(OUTPUT/'01_transformation_measured_pair_counts.png',dpi=160);plt.show()
else:
    print('No measured transformation-related pairs for plotting; descriptive analysis is empty.')

if len(both):
    fig,ax=plt.subplots(figsize=(9,4.8))
    labels=sorted(both['transformation'].unique())
    values=[both.loc[both.transformation==t,'abs_delta_pic50'].dropna().to_numpy() for t in labels]
    values=[v for v in values if len(v)]
    if values:
        labels=[t for t in labels if both.loc[both.transformation==t,'abs_delta_pic50'].notna().any()]
        ax.boxplot(values,tick_labels=labels,showfliers=True)
        ax.axhline(CLIFF_DELTA,linestyle='--',linewidth=1,color='black',label='1 pIC50 unit')
        ax.set_title('Activity differences among proposed transformation pairs')
        ax.set_ylabel('|ΔpIC50| for pairs measured in the same isoform')
        plt.xticks(rotation=15,ha='right');plt.tight_layout()
        plt.savefig(OUTPUT/'02_transformation_activity_differences.png',dpi=160);plt.show()

## 10 · Examine representation sensitivity and exact fingerprint collisions

For each curated pair and detected structure-related pair, compare **ordinary ECFP4**, **chirality-aware ECFP4**, and **RDKit atom-pair fingerprints** (all Tanimoto similarity). This tests how much the numerical definition of “similar” depends on the representation. A bit-vector Tanimoto score of 1.0 is not proof that two chemically distinct structures are equivalent.

In [ ]:
fpgen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048)
fpgen_chiral=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048,includeChirality=True)
atom_pair_gen=rdFingerprintGenerator.GetAtomPairGenerator(fpSize=2048)
fp_bank={
    'ECFP4':[fpgen.GetFingerprint(m) for m in work['_mol']],
    'ECFP4_chiral':[fpgen_chiral.GetFingerprint(m) for m in work['_mol']],
    'atom_pair':[atom_pair_gen.GetFingerprint(m) for m in work['_mol']],
}

comparison=[]
for item in CASE_STUDIES:
    if item['parent'] in name_to_idx and item['product'] in name_to_idx:
        comparison.append({'parent_idx':name_to_idx[item['parent']],
                           'product_idx':name_to_idx[item['product']],
                           'parent':item['parent'],'product':item['product'],
                           'relationship':'curated_'+item['proposed']})
for e in edges_df.itertuples(index=False):
    comparison.append({'parent_idx':e.parent_idx,'product_idx':e.product_idx,
                       'parent':e.parent,'product':e.product,
                       'relationship':e.transformation})
comparison_df=pd.DataFrame(comparison)
if len(comparison_df):
    comparison_df=comparison_df.drop_duplicates(['parent_idx','product_idx','relationship']).reset_index(drop=True)
    for rep,fps in fp_bank.items():
        comparison_df['sim_'+rep]=[
            DataStructs.TanimotoSimilarity(fps[int(a)],fps[int(b)])
            for a,b in zip(comparison_df.parent_idx,comparison_df.product_idx)]
    comparison_df['canonical_exact_match']=[work.loc[a,'_original_canonical']==work.loc[b,'_original_canonical']
                                            for a,b in zip(comparison_df.parent_idx,comparison_df.product_idx)]
    display(comparison_df.loc[comparison_df.relationship.str.startswith('curated')])
    display(comparison_df[['sim_ECFP4','sim_ECFP4_chiral','sim_atom_pair']].describe().round(3))
else:
    print('No comparison pairs available.')
comparison_df.to_csv(OUTPUT/'transformation_representation_comparison.csv',index=False)

## 11 · Optional practical baseline: RF, LightGBM, and Tanimoto kNN

The purpose is to distinguish **limitations of the molecular representation** from limitations of a random forest. This is a deliberately small single-split diagnostic; **do not replace the repeated-split conclusions from Notebook 02 with these results**.

- Scaffold-grouped 80/20 validation per isoform.
- Tanimoto-weighted nearest-neighbor predicts pIC50 using **training labels only**.
- LightGBM and RF use a common ECFP4 bit matrix.
- Experimental confidence-interval widths are used **only for diagnostics**, never as prospective model features.

In [ ]:
model_rows=[]
model_preds=[]
if RUN_MODEL_COMPARISON:
    X=np.zeros((len(work),2048),dtype=np.uint8)
    for i,fp in enumerate(fp_bank['ECFP4']):DataStructs.ConvertToNumpyArray(fp,X[i])
    def scaffold(m):
        core=MurckoScaffold.GetScaffoldForMol(m)
        return Chem.MolToSmiles(core) if core.GetNumAtoms() else 'acyclic_'+Chem.MolToSmiles(m)
    groups=np.array([scaffold(m) for m in work['_mol']],dtype=object)
    def weighted_knn(train_ix,val_ix,y,k=5):
        train_fps=[fp_bank['ECFP4'][int(i)] for i in train_ix]
        ans=[]
        for vi in val_ix:
            sims=np.asarray(DataStructs.BulkTanimotoSimilarity(fp_bank['ECFP4'][int(vi)],train_fps))
            top=np.argsort(sims)[-min(k,len(sims)):]
            w=sims[top]**4
            if w.sum()==0:w=np.ones_like(w)
            ans.append(float(np.dot(w,y[train_ix[top]])/w.sum()))
        return np.asarray(ans)
    for iso in ISO:
        col=f'{iso}_pIC50_direct_inhibition'
        y=pd.to_numeric(work[col],errors='coerce').to_numpy(float)
        ix=np.flatnonzero(np.isfinite(y))
        if len(ix)<12 or len(set(groups[ix]))<2:
            print(f'Skipping modeling {iso}: too few observed structures or scaffold groups')
            continue
        gs=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
        ti,vi=next(gs.split(ix,groups=groups[ix]))
        train_ix,val_ix=ix[ti],ix[vi]
        if len(train_ix)<5 or len(val_ix)<2:continue
        models={
            'RF_ECFP4':RandomForestRegressor(n_estimators=RF_TREES,min_samples_leaf=2,
                                            max_features=0.7,n_jobs=-1,random_state=SEED),
            'LightGBM_ECFP4':LGBMRegressor(n_estimators=GBM_TREES,learning_rate=0.04,
                                          max_depth=-1,num_leaves=15,min_child_samples=max(2,min(15,len(train_ix)//8)),
                                          verbosity=-1,random_state=SEED,n_jobs=-1),
        }
        predictions={'Tanimoto_kNN':weighted_knn(train_ix,val_ix,y,k=5)}
        for name,model in models.items():
            model.fit(X[train_ix],y[train_ix]); predictions[name]=model.predict(X[val_ix])
        for name,pred in predictions.items():
            err=np.abs(y[val_ix]-pred)
            model_rows.append({'isoform':iso,'model':name,'n_train':len(train_ix),'n_valid':len(val_ix),
                               'mae':float(np.mean(err)), 'predicted_sd':float(np.std(pred,ddof=1)),
                               'experimental_sd':float(np.std(y[val_ix],ddof=1))})
            for vi,pr,abse in zip(val_ix,pred,err):
                model_preds.append({'isoform':iso,'model':name,'row_index':int(vi),
                                    'Molecule_Name':work.loc[vi,'Molecule_Name'],
                                    'y_true':float(y[vi]),'y_pred':float(pr),'abs_error':float(abse)})
        print(f'{iso}: trained on {len(train_ix)} structures; evaluated on {len(val_ix)}')
else: print('Model comparison disabled; chemistry analyses complete.')
model_df=pd.DataFrame(model_rows)
model_pred_df=pd.DataFrame(model_preds)
if len(model_df):
    display(model_df)
    fig,ax=plt.subplots(figsize=(9,4.4))
    table=model_df.pivot(index='isoform',columns='model',values='mae')
    table.plot.bar(ax=ax);ax.set_title('Exploratory scaffold-held-out MAE by model');ax.set_ylabel('MAE (pIC50)')
    ax.set_xlabel('CYP isoform');plt.xticks(rotation=0);plt.tight_layout()
    plt.savefig(OUTPUT/'03_model_comparison_mae.png',dpi=160);plt.show()
model_df.to_csv(OUTPUT/'model_comparison_scaffold.csv',index=False)
model_pred_df.to_csv(OUTPUT/'model_comparison_predictions.csv',index=False)

## 12 · Export everything and capture reproducibility metadata

Reproducibility note: We do not claim any PubChem link is a verified metabolite citation, or that a CID alone identifies the biological relationship. The dataset is OpenADMET's public direct-inhibition **training** data. This notebook does not evaluate the blinded challenge test set.

In [ ]:
metadata={
    'notebook':'04_metabolic_transformations_chemical_space.ipynb',
    'dataset_source':LOCAL_CSV or DATA_URL,
    'n_input':int(len(raw)),'n_valid_smiles':int(len(work)),
    'transformations':['aromatic_hydroxylation','methyl_ester_hydrolysis','aryl_O_demethylation'],
    'n_detected_directed_edges':int(len(edges_df)),
    'n_detected_labeled_isoform_edges':int(len(both)),
    'curated_pairs':CASE_STUDIES,
    'fingerprints':['ECFP4','ECFP4_chiral','atom_pair'],
    'seed':SEED, 'run_model_comparison':RUN_MODEL_COMPARISON,
    'rf_trees':RF_TREES,'gbm_trees':GBM_TREES,
    'python':platform.python_version(),'rdkit':rdBase.rdkitVersion,
    'sklearn':__import__('sklearn').__version__,
    'interpretation_caution':'Structural transformation hypothesis; no metabolite identification or prodrug verification.'
}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2))

# A blank provenance template for manually verifying published metabolites or prodrug activity.
literature_template=pd.DataFrame([
  {'proposed_precursor':x['parent'],'proposed_product':x['product'],
   'hypothesized_change':x['proposed'], 'PubChem_CID_or_InChIKey':'',
   'DOI_or_PMID':'','evidence_type':'','assay_or_organism':'',
   'confirmed_metabolite':'unverified','confirmed_prodrug':'unverified','notes':''}
  for x in CASE_STUDIES])
literature_template.to_csv(OUTPUT/'literature_evidence_template.csv',index=False)
(OUTPUT/'README.txt').write_text("CYP Notebook 04 outputs.\n"
    "Only chemical graph relationships are annotated. No metabolite identification performed.\n"
    "PubChem CIDs (if requested) indicate structure records, not metabolite evidence.\n"
    "The CI gap is exploratory, not the official competition metric.\n")
from zipfile import ZipFile, ZIP_DEFLATED
archive=Path('cyp_notebook_04_results.zip')
with ZipFile(archive,'w',ZIP_DEFLATED) as zipf:
    for file in sorted(OUTPUT.iterdir()):
        if file.is_file():zipf.write(file,arcname=f'{OUTPUT.name}/{file.name}')
print('Saved',len(list(OUTPUT.iterdir())),'files. Archive:',archive.resolve())
if os.path.isdir('/content'):
    from google.colab import files
    display(Markdown(f'Output bundle ready: `{archive.name}`. Use the Colab file sidebar or the download prompt below.'))
    # Uncomment for automatic browser download:
    # files.download(str(archive))

## 13 · How to interpret the chemical hypotheses

1. **Check the original compound structures first.** If the curated precursor/product pair fails the exact graph rule, the transformation may involve additional changes (for example, stereochemistry, salt form, another protecting group), or the suggested direction may be wrong. Inspect the PNGs and raw SMILES.
2. **Distinguish exposure from potency.** The ester can be less active in one biochemical assay yet improve permeability in a cellular context, or be an intentionally synthesized analog rather than a prodrug. Neither direct CYP inhibition nor structural similarity identifies the true pharmacologically active species.
3. **CYP versus esterase:** aromatic hydroxylation is *compatible* with CYP oxidation; ester hydrolysis is *compatible* with esterases and generally does not require CYP catalysis. Neither is experimentally established here.
4. **Interpret selection and missingness.** A pair is compared per isoform only if both compounds have quantitative pIC50 measurements. A rare transformation-related labeled pair is not evidence that the chemistry is rare in the full screening library.
5. **Do literature verification using structure identifiers.** Copy the raw SMILES printed in Section 3 into PubChem; check possible synonyms, linked articles, patents, and pharmacology references. Exact structure matching does not establish a parent–metabolite relationship, and absence of references does not disprove one.
6. **Next research step:** If you find literature establishing a compound's parent/metabolite or ester/acid relationship, record the DOI/PMID, assay context, and evidence type in a **manually curated evidence table**. Keep the literature-derived annotations separate from computational transformation rules.

### References
- [OpenADMET public CYP challenge training data](https://huggingface.co/datasets/openadmet/cyp-challenge-train-test)
- [OpenADMET CYP Challenge Tutorial](https://github.com/OpenADMET/CYP-Challenge-Tutorial)
- [PubChem structure search and PUG REST documentation](https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest)